# StockSense Pro — training and evaluation

The supplied model has already completed 30 epochs. This notebook lets you inspect the evidence and reproduce the workflow. Run from a copy of the extracted project. In Colab, upload/unzip the complete project first, then select a GPU runtime if desired. No results are fabricated in this notebook; completed-run metrics live in `reports/`.

In [ ]:
from pathlib import Path
import os
# Change this to your extracted project path in Colab if necessary.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "app.py").exists(), "Set ROOT to the extracted stocksense-pro folder"
os.chdir(ROOT)
print(ROOT)

## 1. Install the training environment

For a CPU-only Linux machine, install CPU PyTorch first as shown in README. On Colab, keep a compatible GPU-enabled PyTorch install. The deployed Streamlit app needs only requirements.txt.

In [ ]:
%pip install -r requirements-training.txt

## 2. Expand and inspect the balanced dataset

There are 100 source images per category, with a fixed 70/15/15 split. Archives retain both original VOC annotations and converted YOLO labels. Normalization and augmentation occur in the model pipeline; test images are not used for training.

In [ ]:
import subprocess, sys, json
subprocess.run([sys.executable, "scripts/unpack_dataset.py"], check=True)
print((ROOT / "dataset/split_summary.json").read_text())
import pandas as pd
manifest = pd.read_csv(ROOT / "dataset/manifest.csv")
assert manifest.pixel_sha256.nunique() == 500
manifest.groupby(["category", "split"]).size().unstack()

In [ ]:
from PIL import Image
from IPython.display import display
for category in ["cereal", "coffee", "juice", "milk", "water"]:
    row = manifest[(manifest.category == category) & (manifest.split == "train")].iloc[0]
    print(category, row["file"])
    display(Image.open(ROOT / "dataset/images/train" / row["file"]))

## 3. Reproduce fine-tuning (optional)

This overwrites models/best.pt and models/best.onnx. The script initializes YOLO11n from COCO, freezes the first 10 layers, and trains for 30 epochs with AdamW, batch 16, 224-pixel input, rotations, brightness, flips and mosaic. Best weights are chosen on validation. Set RUN_TRAINING to True to actually retrain. Use 640-pixel training only when appropriate for your new high-resolution data and compute budget.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    import torch
    device = "0" if torch.cuda.is_available() else "cpu"
    subprocess.run([sys.executable, "scripts/train.py", "--epochs", "30", "--imgsz", "224", "--batch", "16", "--device", device], check=True)
else:
    print("Using the included trained model.")

## 4. Evaluate the exact ONNX pipeline used in the app

Run after the model and confidence setting are finalized. It computes one-to-one box matching, precision, recall, count errors and the confusion matrix on the held-out test split. Low-light and flipped variants are stress checks of these same images, not fresh independent test sets.

In [ ]:
subprocess.run([sys.executable, "scripts/evaluate.py"], check=True)
metrics = json.loads((ROOT / "reports/evaluation.json").read_text())
pd.DataFrame(metrics["per_class"])

In [ ]:
pd.DataFrame(metrics["stress_tests"])

In [ ]:
display(Image.open(ROOT / "reports/confusion_matrix.png"))
errors = pd.read_csv(ROOT / "reports/counting_results.csv")
errors.sort_values("absolute_category_error", ascending=False).head(10)

## 5. Inspect a failure

Compare visible objects with supplied labels and predictions. Some dense backgrounds are incompletely labeled. Do not assume every discrepancy is solely a model error; manually audit ambiguous cases before making inventory claims.

In [ ]:
worst = errors.sort_values("absolute_category_error", ascending=False).iloc[0]
print(worst)
display(Image.open(ROOT / "dataset/images/test" / worst["image"]))

## 6. Test and deploy

Upload the project contents to GitHub, then select app.py and Python 3.12 in Streamlit Community Cloud. See START_HERE.md. Add your real repository and deployed URLs to README and the submission details JSON. Complete genuine human feedback after deployment.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], check=True)

## Explain the work

Be ready to explain detection vs classification, normalized bounding boxes, confidence vs IoU, train/validation/test separation, the count-3 threshold decision, class imbalance, false positives/negatives, mAP vs count accuracy, and why a snapshot cannot measure sales velocity. Read the source code and acknowledge Codex assistance according to your school policy.

References and dataset/model attribution are in README.md and DATASET_CARD.md.